# 闭包、装饰器与深浅拷贝

本篇对应《Python进阶》课程第 3 天，围绕三个主题展开：闭包、装饰器，以及可变对象相关的浅拷贝与深拷贝。

学习重点：

- 理解闭包的构成条件，以及 `nonlocal` 对外部函数变量的修改作用。
- 掌握装饰器的本质、语法糖、参数传递和返回值传递。
- 能够读懂多个装饰器和带参数装饰器的执行过程。
- 区分变量赋值、浅拷贝和深拷贝，避免嵌套可变对象被意外共享。

课程代码按讲义顺序保留在 Markdown 代码围栏中，示例不在本笔记中执行。

## 一、闭包

### 1. 闭包解决的问题

函数执行结束后，函数内部的局部变量通常会随着调用栈释放。如果希望后续调用继续使用某个外部函数中的数据，可以让内部函数引用外部函数的变量，并把内部函数返回出去。这样，内部函数就能“记住”创建它时的环境。

闭包常用于保存状态、延迟执行和为装饰器提供基础。

### 2. 闭包的语法结构

闭包需要同时满足三个条件：

1. **有嵌套**：一个函数内部定义另一个函数；
2. **有引用**：内部函数使用外部函数的变量或参数；
3. **有返回**：外部函数返回内部函数名，而不是调用内部函数。

基本结构如下：

```python
def 外部函数名(外部参数):
    def 内部函数名(内部参数):
        # 使用外部参数或外部函数中的变量
        ...
    return 内部函数名
```

返回 `内部函数名` 表示把函数对象交给调用方；写成 `return 内部函数名()` 则会立即调用它，含义不同。

### 3. 使用闭包保存外部变量

下面的闭包把 `num1` 保存在外部函数返回的内部函数中。调用 `func_out(10)` 后，得到的 `f` 仍然可以在之后使用 `10`：

```python
def func_out(num1):
    def func_inner(num2):
        num = num1 + num2
        print("现在的值：", num)

    return func_inner


f = func_out(10)
f(1)
```

执行过程可以按两步理解：

1. `func_out(10)` 创建内部函数，并把外部参数 `num1` 的值保存到闭包环境中；
2. `f(1)` 调用内部函数，内部函数把保存的 `num1` 与本次传入的 `num2` 相加。

闭包保存的是变量所在的环境，而不是简单复制一段代码。

### 4. `nonlocal`：修改外部函数变量

`global` 用于声明要操作全局作用域中的变量；`nonlocal` 用于让内部函数修改外部函数作用域中的变量。没有 `nonlocal` 时，内部函数中对同名变量赋值会被当作创建新的局部变量。

```python
def func_out():
    a = 100

    def func_inner():
        nonlocal a
        a = a + 1
        print(f"修改的结果:{a}")

    return func_inner


number = func_out()
number()
```

`nonlocal a` 不会创建全局变量，它只会向外查找最近的函数作用域。若外层没有名为 `a` 的变量，使用 `nonlocal a` 会报错。

## 二、装饰器

### 1. 装饰器的作用和本质

装饰器用于在**不修改原函数源码和调用方式**的前提下，为原函数增加额外功能，例如登录验证、日志记录、权限检查或友好提示。

装饰器本质上是一个闭包，但比普通闭包多了一项用途：内部函数要在调用原函数的基础上增加功能，并返回新的函数对象。典型构成是：

- 外部函数接收被装饰的原函数；
- 内部函数完成额外功能，再调用原函数；
- 外部函数返回内部函数。

### 2. 传统写法

传统方式先定义装饰器，再显式执行 `comment = check(comment)`：

```python
def check(fn):
    def inner():
        print("登录验证。。。")
        fn()

    return inner


def comment():
    print("发表评论")


comment = check(comment)
comment()
```

赋值之后，变量 `comment` 指向的是包装函数 `inner`。调用 `comment()` 时，会先执行验证逻辑，再由 `inner` 调用原来的 `comment` 函数。

### 3. `@装饰器` 语法糖

Python 提供了更直观的写法：

```python
def check(fn):
    def inner():
        print("请先登陆")
        fn()

    return inner


@check
def comment():
    print("发表评论")


comment()
```

解释器定义完 `comment` 后，会立即把它改写为近似下面的形式：

```python
comment = check(comment)
```

因此，装饰器是在函数定义阶段完成包装；真正调用 `comment()` 时，执行的是包装后的函数。

## 三、装饰器适配不同类型的函数

装饰器内部函数的参数列表和返回方式，必须与原函数的调用约定保持一致：

- 原函数没有参数，内部函数通常也不接收业务参数；
- 原函数有参数，内部函数要接收并继续传给原函数；
- 原函数有返回值，内部函数要返回原函数的结果；
- 同时有参数和返回值时，两点都要满足。

### 1. 无参、无返回值

```python
def print_info(func):
    def inner():
        print("【友好提示】正在努力计算中...")
        func()

    return inner


@print_info
def get_sum():
    a = 10
    b = 20
    c = a + b
    print(f"两数之和为:{c}")


get_sum()
```

内部函数先输出提示，再调用原函数；原函数没有返回值，所以内部函数也不需要返回结果。

### 2. 有参、无返回值

原函数有参数时，包装函数要接收同样的参数，并把参数传递给原函数：

```python
def print_info(func):
    def inner(a, b):
        print("【友好提示2】努力计算中...")
        func(a, b)

    return inner


@print_info
def get_sum(x, y):
    z = x + y
    print(f"两数之和为:{z}")


get_sum(20, 35)
```

如果 `inner` 不接收 `a`、`b`，调用装饰后的 `get_sum(20, 35)` 就会出现参数数量不匹配。

### 3. 无参、有返回值

原函数返回的结果不能在包装过程中丢失，需要用变量接收并返回：

```python
def print_info(func):
    def inner():
        print("【友好提示】正在努力计算中...")
        result = func()
        return result

    return inner


@print_info
def get_sum():
    x = 19
    y = 34
    z = x + y
    return z


result = get_sum()
```

### 4. 有参、有返回值

```python
def print_info(func):
    def inner(a, b):
        print("【友好提示】正在努力计算中...")
        result = func(a, b)
        return result

    return inner


@print_info
def get_sum(x, y):
    z = x + y
    return z


result = get_sum(13, 45)
```

这四种情况可以归纳为一句话：**包装函数的参数和返回值要与原函数的约定一致。**

### 5. 通用装饰器：`*args` 与 `**kwargs`

当原函数的参数数量或形式不固定时，可以用不定长参数编写通用装饰器：

- `*args` 接收位置参数，类型是元组；
- `**kwargs` 接收关键字参数，类型是字典；
- 调用原函数时使用 `func(*args, **kwargs)` 解包回去。

```python
def print_info(func):
    def inner(*args, **kwargs):
        print("【友好提示】正在努力计算中...")
        return func(*args, **kwargs)

    return inner


@print_info
def get_sum(*args, **kwargs):
    result = 0
    for value in args:
        result += value
    for value in kwargs.values():
        result += value
    return result


result = get_sum(1, 3, 4, 5, a=10, b=20, c=30)
```

通用装饰器既保留额外功能，也能适配不同参数形式。实际项目中还可以使用 `functools.wraps` 保留原函数的名称和文档信息，本课重点先放在闭包和参数传递上。

## 四、多个装饰器

一个函数可以同时使用多个装饰器。例如发表评论前，先输入用户名和密码，再完成验证码检查：

```python
def check_user(fn):
    def inner():
        print("输入用户名和密码")
        fn()

    return inner


def check_code(fn):
    def inner():
        print("登录验证码")
        fn()

    return inner


@check_user
@check_code
def comment():
    print("发表评论")


comment()
```

装饰顺序遵循从下到上的构造规则：离原函数最近的 `@check_code` 先执行，结果再交给 `@check_user` 包装。等价写法是：

```python
comment = check_user(check_code(comment))
```

调用时则从最外层包装函数开始，依次进入内层包装函数，最后执行原函数。

## 五、带参数的装饰器

### 1. 为什么需要三层函数

普通装饰器的外层函数接收原函数；带参数装饰器还要先接收装饰器自己的配置参数。例如用 `"+"` 或 `"-"` 指定提示内容：

```python
def logging(flag):
    def decorator(fn):
        def inner(num1, num2):
            if flag == "+":
                print("--正在努力加法计算--")
            elif flag == "-":
                print("--正在努力减法计算--")
            result = fn(num1, num2)
            return result

        return inner

    return decorator
```

这里有三层职责：

1. `logging(flag)` 接收装饰器参数；
2. `decorator(fn)` 接收要装饰的原函数；
3. `inner(num1, num2)` 接收原函数调用时的业务参数，并增加提示、调用原函数、返回结果。

### 2. 使用带参数装饰器

```python
@logging("+")
def add(a, b):
    return a + b


@logging("-")
def reduce(a, b):
    return a - b


add_result = add(1, 3)
reduce_result = reduce(5, 2)
```

`@logging("+")` 的执行可以展开为：

```python
add = logging("+")(add)
```

而 `logging("+")` 的返回值是 `decorator`，所以整体仍然符合普通装饰器“接收原函数并返回包装函数”的规则。带参数装饰器的关键是不要把装饰器参数和原函数参数混在同一层。

## 六、可变类型、不可变类型与变量引用

### 1. 不可变类型和可变类型

不可变对象创建后，其内容不能原地修改，常见类型包括：字符串、元组、整数和浮点数。对不可变对象做“修改”时，Python 通常会创建新对象，再让变量指向新对象。

可变对象可以原地修改，常见类型包括：列表、字典和集合。修改可变对象的内容时，对象本身的地址通常不变。

“可变”描述的是对象能否原地改变；变量本身只是对象的引用。

### 2. 赋值本质是引用

变量赋值不会自动复制对象，而是让另一个变量指向同一个对象：

```python
numbers = [1, 2, 3]
other = numbers
other.append(4)
```

此时 `numbers` 和 `other` 引用同一个列表，修改 `other` 也会影响通过 `numbers` 看到的内容。可以使用 `id()` 观察两个变量是否引用同一对象：

```python
print(id(numbers) == id(other))
```

因此，想要得到独立的列表副本时，不能只使用普通赋值。

## 七、浅拷贝

### 1. 浅拷贝的含义

浅拷贝会创建一个新的最外层对象，但不会递归复制内部嵌套对象。最外层容器与原对象地址不同；容器中的元素引用通常仍然共享。

可使用 `copy.copy()`，也可以使用列表切片等方式创建浅拷贝：

```python
import copy

original = [1, 2, [3, 4]]
shallow = copy.copy(original)

shallow.append(5)
shallow[2].append(6)
```

`shallow.append(5)` 只修改了新的最外层列表；`shallow[2].append(6)` 修改的是共享的内层列表，因此原对象中的嵌套列表也会受到影响。

### 2. 浅拷贝的判断规律

- 浅拷贝不可变对象时，通常与赋值效果接近，`id()` 可能相同，因为不可变对象无需重新创建；
- 浅拷贝可变对象时，最外层对象通常拥有不同地址；
- 如果对象内部没有嵌套可变对象，浅拷贝往往已经够用；
- 如果内部包含列表、字典等嵌套可变对象，修改嵌套内容仍可能相互影响。

判断拷贝是否足够时，重点检查数据结构中是否存在需要独立复制的嵌套可变对象。

## 八、深拷贝

### 1. 深拷贝的含义

深拷贝会递归复制对象及其嵌套元素，得到与原对象相互独立的完整副本。修改原对象或副本中的嵌套可变对象，都不会影响另一方。

使用 `copy` 模块中的 `deepcopy()`：

```python
import copy

original = [1, 2, [3, 4], {"name": "Python"}]
deep = copy.deepcopy(original)

deep[2].append(5)
deep[3]["level"] = "advanced"
```

这里的 `deep` 会递归复制内层列表和字典，所以修改它们不会改变 `original`。对于不可变元素，深拷贝没有必要重新开辟空间，仍可能复用同一个不可变对象，这是正常现象。

### 2. 三种方式对比

| 方式 | 最外层对象 | 嵌套可变对象 | 典型风险 |
| --- | --- | --- | --- |
| 直接赋值 `b = a` | 共享 | 共享 | 任意修改都会互相影响 |
| 浅拷贝 `copy.copy(a)` | 通常独立 | 共享 | 修改嵌套对象会影响原对象 |
| 深拷贝 `copy.deepcopy(a)` | 独立 | 递归独立 | 占用更多内存，复制成本更高 |

选择原则：只需要复制一层时使用浅拷贝；需要完整隔离嵌套数据时使用深拷贝；如果对象本身不可变或不需要修改，可以直接共享引用。

## 九、要点回顾与易错点

### 闭包

- 闭包必须有嵌套、引用和返回三个条件；
- 外部函数返回内部函数名，不能误写成调用结果；
- `nonlocal` 修改的是最近的外部函数变量，`global` 修改的是全局变量。

### 装饰器

- 装饰器在不改原函数源码和调用方式的前提下增加功能；
- `@decorator` 等价于定义完成后的 `func = decorator(func)`；
- 包装函数的参数和返回值要与原函数约定一致；
- 不定长参数装饰器使用 `*args` 和 `**kwargs` 适配不同调用方式；
- 多个装饰器从靠近原函数的装饰器开始构造；
- 带参数装饰器通常需要“参数函数 → 装饰器 → 内部函数”三层结构。

### 拷贝

- 变量赋值只是增加一个引用，不会复制对象；
- 浅拷贝只复制最外层，嵌套可变对象仍可能共享；
- 深拷贝递归复制嵌套对象，适合需要完全隔离的数据；
- 不可变对象和可变对象的拷贝表现不同，分析时要结合数据结构层级。